# 06 - Etiquetado de Tokens (Labeling)

**Objetivo**: Clasificar cada token como gem, moderate_success, neutral, failure o rug.

### Sistema de clasificacion:

| Label | Criterio |
|-------|----------|
| **gem** | Alcanzo 10x Y se mantuvo >5x por 7+ dias |
| **moderate_success** | Alcanzo 3x-10x Y se mantuvo >2x por 3+ dias |
| **neutral** | Precio entre 0.3x y 3x |
| **failure** | Precio cayo bajo 0.1x |
| **rug** | Precio cayo bajo 0.01x en 72h O liquidez retirada 90%+ |

### Clasificacion binaria:
- **success (1)**: Alcanzo 5x en 30 dias
- **failure (0)**: Nunca alcanzo 5x

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from src.data.storage import Storage
from src.models.labeler import Labeler
import config

storage = Storage()
print(f"Tokens en BD: {storage.stats().get('tokens', 0)}")
print(f"OHLCV en BD: {storage.stats().get('ohlcv', 0)}")

## Ejecutar labeling

In [ ]:
# Crear el labeler y clasificar todos los tokens
labeler = Labeler(storage=storage)
labels_df = labeler.label_all_tokens()

print(f"\nTokens etiquetados: {len(labels_df)}")
if not labels_df.empty:
    display(labels_df.head(10))

In [ ]:
# Distribucion de labels multiclase
if not labels_df.empty and 'label_multi' in labels_df.columns:
    label_counts = labels_df['label_multi'].value_counts()
    print("Distribucion de labels (multiclase):")
    print(label_counts)
    print(f"\nPorcentajes:")
    print((label_counts / len(labels_df) * 100).round(1))
    
    # Grafico
    colors = {
        'gem': '#00C853',
        'moderate_success': '#2196F3',
        'neutral': '#9E9E9E',
        'failure': '#FF5722',
        'rug': '#212121',
    }
    fig = px.bar(
        x=label_counts.index,
        y=label_counts.values,
        title='Distribucion de Labels (Multiclase)',
        labels={'x': 'Label', 'y': 'Cantidad'},
        color=label_counts.index,
        color_discrete_map=colors,
    )
    fig.show()

In [ ]:
# Distribucion binaria
if not labels_df.empty and 'label_binary' in labels_df.columns:
    binary_counts = labels_df['label_binary'].value_counts()
    print("Distribucion binaria:")
    print(f"  Success (1): {binary_counts.get(1, 0)} ({binary_counts.get(1, 0)/len(labels_df)*100:.1f}%)")
    print(f"  Failure (0): {binary_counts.get(0, 0)} ({binary_counts.get(0, 0)/len(labels_df)*100:.1f}%)")
    print(f"\nRatio desbalance: 1:{binary_counts.get(0, 1)/max(binary_counts.get(1, 1), 1):.0f}")

In [ ]:
# Distribucion de max_multiple alcanzado
if not labels_df.empty and 'max_multiple' in labels_df.columns:
    max_mult = labels_df['max_multiple'].dropna()
    max_mult = max_mult[max_mult > 0]
    
    fig = px.histogram(
        x=np.log10(max_mult),
        nbins=50,
        title='Distribucion del Maximo Multiple Alcanzado (log10)',
        labels={'x': 'log10(Max Multiple)', 'y': 'Frecuencia'},
    )
    # Lineas de referencia
    for mult, label in [(np.log10(5), '5x'), (np.log10(10), '10x')]:
        fig.add_vline(x=mult, line_dash='dash', annotation_text=label)
    fig.show()
    
    print(f"Mediana max_multiple: {max_mult.median():.2f}x")
    print(f"% que alcanzo 5x: {(max_mult >= 5).mean()*100:.1f}%")
    print(f"% que alcanzo 10x: {(max_mult >= 10).mean()*100:.1f}%")

## Guardar labels

In [ ]:
if not labels_df.empty:
    # Guardar en Parquet
    labels_df.to_parquet(config.PROCESSED_DIR / "labels.parquet", index=False)
    print(f"Labels guardados: {config.PROCESSED_DIR / 'labels.parquet'}")
    print(f"Total: {len(labels_df)} tokens etiquetados")
    print("\nSiguiente paso: notebook 07_modeling.ipynb")
else:
    print("No se generaron labels. Necesitas datos OHLCV (ejecuta notebooks 02-03).")